In [1]:
import tensorflow as tf


2025-10-23 18:25:13.457705: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-23 18:25:13.548084: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-23 18:25:13.572492: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-23 18:25:14.067772: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:

# ---------------------------
# Parameters
# ---------------------------
batch_size = 4
signal_length = 64
window_size = 20
stride = 4
noise_std = 0.05
hidden_units = 32

# ---------------------------
# Dummy batch
# ---------------------------
x = tf.random.normal((batch_size, signal_length, 1))
x_noisy = x + noise_std * tf.random.normal(tf.shape(x))

# ---------------------------
# Trainable orthogonal transform T via QR
# ---------------------------
W = tf.Variable(tf.random.normal((window_size, window_size)), trainable=True)

# ---------------------------
# Learnable harmonic aggregator (NN)
# ---------------------------
harmonic_nn = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(window_size,)),
    tf.keras.layers.Dense(hidden_units, activation='relu'),
    tf.keras.layers.Dense(window_size)  # output same size as harmonic vector
])

# ---------------------------
# Optimizer
# ---------------------------
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

# ---------------------------
# Forward + loss
# ---------------------------
def forward_and_loss(W, x_noisy, x_true):
    # QR decomposition for orthogonal T
    Q, _ = tf.linalg.qr(W)
    T = Q  # orthogonal

    # Conv1D: sliding window
    conv_kernel = tf.reshape(T, (window_size, 1, window_size))
    y = tf.nn.conv1d(x_noisy, filters=conv_kernel, stride=stride, padding='VALID')
    batch, n_windows, n_harmonics = y.shape

    # Normalized averaging across windows
    harmonic_avg = tf.reduce_mean(y, axis=1)  # [batch, window_size]

    # Pass through learnable NN to combine harmonics
    harmonic_processed = harmonic_nn(harmonic_avg)  # [batch, window_size]

    # Reconstruct signal via T^T
    x_reconstructed = tf.matmul(harmonic_processed, tf.transpose(T))  # [batch, window_size]

    # Slice to match original signal length if needed
    x_target = x_true[:, :x_reconstructed.shape[1], 0]

    # Loss: only reconstruction
    loss_recon = tf.reduce_mean((x_target - x_reconstructed)**2)
    return loss_recon, x_reconstructed

# ---------------------------
# Training step
# ---------------------------
with tf.GradientTape() as tape:
    loss, x_hat = forward_and_loss(W, x_noisy, x)
trainable_vars = [W] + harmonic_nn.trainable_variables
grads = tape.gradient(loss, trainable_vars)
optimizer.apply_gradients(zip(grads, trainable_vars))

print("Loss:", loss.numpy())


I0000 00:00:1761258323.008886   20349 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1761258323.976168   20349 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1761258323.976916   20349 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1761258323.985882   20349 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

Loss: 0.8172687
